## Ingest BTCUSDT AggTrades → Delta Table

Read Binance `aggTrades` CSV from a Unity Catalog Volume via **Spark CSV**, and write as a managed Delta table with **liquid clustering**.

| Parameter | Value |
|---|---|
| **Source** | `/Volumes/smart-trading-dev/00_volume/raw-btcusdt-aggtrades/BTCUSDT-aggTrades-2025-03.csv` |
| **Target** | `` `smart-trading-dev`.`01_bronze`.`aggtrades` `` |
| **Pipeline** | CSV → `spark.read.csv` → Transform → Delta |
| **Clustering** | Liquid clustering on `trade_date` |

In [0]:
# --- Source (Volume) ---
CATALOG = "smart_trading_dev"
SCHEMA  = "00_volume"
VOLUME  = "raw_btcusdt_aggtrades"
CSV_FILE_NAME = "BTCUSDT-aggTrades-2025-03.csv"

CSV_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{CSV_FILE_NAME}"

# --- Target (Delta table) ---
TBL_SCHEMA = "01_bronze"
TBL_NAME   = "aggtrades"
TGT_TBL_PATH = f"`{CATALOG}`.`{TBL_SCHEMA}`.`{TBL_NAME}`"

# --- Data ---
SYMBOL = "BTCUSDT"

print(f"Source:  {CSV_PATH}")
print(f"Target:  {TGT_TBL_PATH}")
print(f"Symbol:  {SYMBOL}")

In [0]:
# Quick preview of the CSV file
df_preview = spark.read.csv(
    CSV_PATH,
    header=False,
    inferSchema=True
).limit(5)

print(f"Columns: {len(df_preview.columns)}")
print(f"Preview (first 5 rows):")
display(df_preview)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, BooleanType

print(f"Reading CSV from: {CSV_PATH}")
print(f"Target table: {TGT_TBL_PATH}\n")

# Ensure catalog & schema exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS `{CATALOG}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{TBL_SCHEMA}`")

# Define schema for the aggTrades CSV
schema = StructType([
    StructField("agg_trade_id", LongType(), False),
    StructField("price", DoubleType(), False),
    StructField("quantity", DoubleType(), False),
    StructField("first_trade_id", LongType(), False),
    StructField("last_trade_id", LongType(), False),
    StructField("transact_time", LongType(), False),
    StructField("is_buyer_maker", BooleanType(), False),
    StructField("is_best_match", BooleanType(), False)
])

print("Reading CSV with Spark...")
df = spark.read.csv(
    CSV_PATH,
    schema=schema,
    header=False
)

row_count = df.count()
print(f"✓ Read {row_count:,} rows from CSV\n")

# Add derived columns (transact_time is in microseconds)
df_transformed = (
    df
    .withColumn("symbol", F.lit(SYMBOL))
    .withColumn("transact_timestamp", F.timestamp_micros("transact_time"))
    .withColumn("trade_date", F.to_date("transact_timestamp"))
)

print("Writing to Delta table with liquid clustering on trade_date...")
(
    df_transformed
    .writeTo(TGT_TBL_PATH)
    .using("delta")
    .clusterBy("trade_date")
    .createOrReplace()
)

final_count = spark.table(TGT_TBL_PATH).count()
print(f"\n✓ Total rows written: {final_count:,}")
print(f"✓ Delta table: {TGT_TBL_PATH}")
print(f"✓ Liquid clustering: trade_date")

In [0]:
# Verify the table was created and data loaded correctly
result_df = spark.table(TGT_TBL_PATH)

print(f"Table: {TGT_TBL_PATH}")
print(f"Row count: {result_df.count():,}")

date_range = result_df.select(F.min('trade_date'), F.max('trade_date')).first()
print(f"Date range: {date_range[0]} to {date_range[1]}")

print(f"\nSample rows:")
display(result_df.limit(10))

In [0]:
# No cleanup needed - CSV read directly via Spark
print("✓ No temporary files to clean up")